# Public repository version

All executed outputs were cleared before public release. Dataset previews were removed, and exported prediction files exclude the original Arabic text.


In [ ]:
!pip uninstall -y numpy scikit-learn
!pip install -q numpy==1.26.4 scikit-learn==1.5.2
!pip install -q -U transformers datasets accelerate pandas openpyxl matplotlib

In [ ]:
import numpy as np
import sklearn
import transformers
import torch
import pandas as pd

print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("pandas:", pd.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import transformers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# =========================
# CONFIG
# =========================
SEED = 42
MODEL_NAME = "UBC-NLP/MARBERT"

TEXT_COL = "text"
LABEL_COL = "emotion"

MAX_LEN = 128
OUTPUT_DIR = "/content/marbert-output"
SAVE_DIR = "/content/final_marbert_model"

DATA_PATH = "/content/EmotionsFile_GoldLabelCSV.csv"

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("Seed set to:", SEED)

In [ ]:
if DATA_PATH.endswith(".csv"):
    df = pd.read_csv(DATA_PATH)
elif DATA_PATH.endswith(".xlsx"):
    df = pd.read_excel(DATA_PATH)
else:
    raise ValueError("Unsupported file format. Please use CSV or XLSX.")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
required_cols = [TEXT_COL, LABEL_COL]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

print("Columns OK")

In [ ]:
df = df[[TEXT_COL, LABEL_COL]].copy()
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)

print("After filtering:", df.shape)

In [ ]:
def clean_arabic_text(text):
    text = str(text).strip()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df[TEXT_COL] = df[TEXT_COL].astype(str).apply(clean_arabic_text)

print("Text cleaned")

In [ ]:
print(df[LABEL_COL].value_counts())
print("\nNumber of classes:", df[LABEL_COL].nunique())

In [ ]:
label_list = sorted(df[LABEL_COL].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df[LABEL_COL].map(label2id)

print("label_list:", label_list)
print("label2id:", label2id)

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
print("Train distribution:")
print(train_df[LABEL_COL].value_counts())

print("\nValidation distribution:")
print(val_df[LABEL_COL].value_counts())

print("\nTest distribution:")
print(test_df[LABEL_COL].value_counts())

In [ ]:
marbert_train_df = train_df[[TEXT_COL, "label_id"]].copy()
marbert_val_df   = val_df[[TEXT_COL, "label_id"]].copy()
marbert_test_df  = test_df[[TEXT_COL, "label_id"]].copy()

marbert_train_df.columns = ["text", "label"]
marbert_val_df.columns   = ["text", "label"]
marbert_test_df.columns  = ["text", "label"]


In [ ]:
train_dataset = Dataset.from_pandas(marbert_train_df, preserve_index=False)
val_dataset   = Dataset.from_pandas(marbert_val_df, preserve_index=False)
test_dataset  = Dataset.from_pandas(marbert_test_df, preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset_dict

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("MARBERT loaded successfully")

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LEN
    )

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
print("Tokenization done")
tokenized_datasets

In [ ]:
print(tokenized_datasets["train"][0])

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Data collator ready")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=False,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    disable_tqdm=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Fresh trainer created")

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Validation
val_pred_output = trainer.predict(tokenized_datasets["validation"])
val_preds = np.argmax(val_pred_output.predictions, axis=-1)
val_labels = val_pred_output.label_ids

val_acc = accuracy_score(val_labels, val_preds)
val_macro_f1 = f1_score(val_labels, val_preds, average="macro")
val_weighted_f1 = f1_score(val_labels, val_preds, average="weighted")

print("Validation Accuracy:", round(val_acc, 4))
print("Validation Macro-F1:", round(val_macro_f1, 4))
print("Validation Weighted-F1:", round(val_weighted_f1, 4))

# Test
test_pred_output = trainer.predict(tokenized_datasets["test"])
test_preds = np.argmax(test_pred_output.predictions, axis=-1)
test_labels = test_pred_output.label_ids

test_acc = accuracy_score(test_labels, test_preds)
test_macro_f1 = f1_score(test_labels, test_preds, average="macro")
test_weighted_f1 = f1_score(test_labels, test_preds, average="weighted")

print("\nTest Accuracy:", round(test_acc, 4))
print("Test Macro-F1:", round(test_macro_f1, 4))
print("Test Weighted-F1:", round(test_weighted_f1, 4))

In [ ]:
pred_output = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

print(classification_report(
    labels,
    preds,
    target_names=[id2label[i] for i in range(len(id2label))],
    digits=4
))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(labels, preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[id2label[i] for i in range(len(id2label))]
)

fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("MARBERT Confusion Matrix")
plt.show()

In [ ]:
results_df = pd.DataFrame({
    "dataset_index": test_df.index,
    "gold_label": labels,
    "prediction": preds
})

results_df["gold_label"] = results_df["gold_label"].map(id2label)
results_df["prediction"] = results_df["prediction"].map(id2label)
results_df["correct"] = (results_df["gold_label"] == results_df["prediction"]).astype(int)

results_df.to_csv("/content/marbert_test_predictions.csv", index=False)
print("Saved privacy-preserving predictions to /content/marbert_test_predictions.csv")
print("Rows saved:", len(results_df))


In [ ]:
results_df = pd.DataFrame({
    "dataset_index": test_df.index,
    "gold_label": labels,
    "prediction": preds
})

results_df["gold_label"] = results_df["gold_label"].map(id2label)
results_df["prediction"] = results_df["prediction"].map(id2label)
results_df["correct"] = (results_df["gold_label"] == results_df["prediction"]).astype(int)

results_df.to_csv("/content/marbert_test_predictions.csv", index=False)
print("Saved privacy-preserving predictions to /content/marbert_test_predictions.csv")
print("Rows saved:", len(results_df))


In [ ]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("MARBERT model saved to:", SAVE_DIR)

In [ ]:
def predict_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=-1).item()

    return id2label[pred_id]

print(predict_text("أنا سعيدة جدا اليوم"))
print(predict_text("أنا غاضبة جدًا من هذا التصرف"))
print(predict_text("أشعر بالخوف من المستقبل"))
print(predict_text("أنا حزينة اليوم"))